# XGBoost with Hierarchical Architecture

Without ECOC (Hierarchical Only): Each specialist is a single Multi-class XGBoost model. 

For example, if the Gater sends a sample to the "Insect" room, the Insect Specialist simply looks at all 28 insect species and says, "I'm 80% sure this is species A and 20% sure it's species B."

In [18]:
import numpy as np
import joblib
import pandas as pd
from model.taxonomy_gater import train_taxonomy_gater
from model.ecoc_specialist import DummySpecialist, train_specialist
from model.pipeline import hybrid_predict_proba
from model.threshold_opt import evaluate_optimized_on_probs, get_optimal_thresholds
from utils import create_taxonomy_map, extract_taxonomy_labels, load_data

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load and Prepare Data

In [ ]:
TAXONOMY_PATH = '../../data/taxonomy.csv'
DATA_PATH = '../../data/processed/final_features.csv'

X_train, y_train, X_val, y_val, X_test, y_test, species_list = load_data(data_dir=DATA_PATH)
print(f"First 5 species in list: {species_list[:5]}")

tax_map = create_taxonomy_map(species_list, mapping_file=TAXONOMY_PATH)
    
y_tax_train = extract_taxonomy_labels(y_train, species_list, mapping_file=TAXONOMY_PATH)
print(f"Unique taxonomy labels: {np.unique(y_tax_train)}")
y_tax_val = extract_taxonomy_labels(y_val, species_list, mapping_file=TAXONOMY_PATH)
y_tax_test = extract_taxonomy_labels(y_test, species_list, mapping_file=TAXONOMY_PATH)

print(len(X_train), len(y_train), len(X_val), len(y_val), len(X_test), len(y_test))

Loading data from ../../data/processed/final_features.csv...
Detected 234 unique species.
First 5 species in list: ['1161364', '116570', '1176823', '1491113', '1595929']
Unique taxonomy labels: [0 1 2 3 4]
135012 135012 28932 28932 28932 28932


### Train the Top-Level Gater

In [14]:
gater = train_taxonomy_gater(X_train, y_tax_train, X_val, y_tax_val)

Gater: Training with 'balanced' sample weights...


### Train Specialists for all 5 classes

In [ ]:
specialists = {}
taxonomy_names = ['Insecta', 'Reptilia', 'Amphibia', 'Mammalia', 'Aves']

for tax_id, name in enumerate(taxonomy_names):
    print(f"\n--- Training Specialist for: {name} (ID: {tax_id}) ---")
        
    mask_train = (y_tax_train == tax_id)
        
    if np.any(mask_train):
        X_sub = X_train[mask_train]
        
        target_cols = tax_map[tax_id]
        y_sub_filtered = y_train[mask_train][:, target_cols]
        
        num_expected = len(target_cols) 
            
        if len(X_sub) < 5: 
            print(f"Warning: {name} has too few samples ({len(X_sub)}). Using Dummy Specialist.")
            specialists[tax_id] = DummySpecialist(num_classes=num_expected)
        else:
            specialists[tax_id] = train_specialist(X_sub, y_sub_filtered, num_expected)
    else:
        target_cols = tax_map[tax_id]
        specialists[tax_id] = DummySpecialist(num_classes=len(target_cols))


--- Training Specialist for: Insecta (ID: 0) ---

--- Training Specialist for: Reptilia (ID: 1) ---

--- Training Specialist for: Amphibia (ID: 2) ---

--- Training Specialist for: Mammalia (ID: 3) ---

--- Training Specialist for: Aves (ID: 4) ---


### Final Hybrid Inference

In [ ]:
print("\nRunning Final Hybrid Inference...")
# 1. Get probabilities for both sets
val_probs = hybrid_predict_proba(X_val, gater, specialists, tax_map)
test_probs = hybrid_predict_proba(X_test, gater, specialists, tax_map)

print("Optimizing Per-Class Thresholds...")
best_thresholds = get_optimal_thresholds(y_val, val_probs)

val_f1, val_lrap = evaluate_optimized_on_probs(val_probs, y_val, best_thresholds, "Validation")

final_f1, final_lrap = evaluate_optimized_on_probs(test_probs, y_test, best_thresholds, "Test (Final)")


Running Final Hybrid Inference...
Optimizing Per-Class Thresholds...

Results on Validation (Optimized):
Macro F1 : 0.4843
LRAP     : 0.5830

Results on Test (Final) (Optimized):
Macro F1 : 0.4495
LRAP     : 0.5835


In [31]:
val_f1_static, val_lrap_static = evaluate_optimized_on_probs(val_probs, y_val, None, "Validation")


Results on Validation (Static (0.3)):
Macro F1 : 0.4519
LRAP     : 0.5830


In [27]:
print("\nRunning Final Hybrid Inference on Test Set...")
probs = hybrid_predict_proba(X_test, gater, specialists, tax_map)

print("Optimizing Per-Class Thresholds...")
best_thresholds = get_optimal_thresholds(y_val, probs)

test_probs = hybrid_predict_proba(X_test, gater, specialists, tax_map)

final_f1, final_lrap = evaluate_optimized_on_probs(test_probs, y_test, best_thresholds, "Test (Final)")


Running Final Hybrid Inference on Test Set...
Optimizing Per-Class Thresholds...

Results on Test (Final) (Optimized):
Macro F1 : 0.3499
LRAP     : 0.5835


In [ ]:
from sklearn.metrics import classification_report

y_pred_binary = (test_probs > best_thresholds).astype(int)

report = classification_report(
    y_test, 
    y_pred_binary, 
    target_names=species_list, 
    zero_division=0
)

print("\n--- Detailed Species-Level Classification Report ---")
print(report)

with open('output/species_classification_report.txt', 'w') as f:
    f.write(report)


--- Detailed Species-Level Classification Report ---
              precision    recall  f1-score   support

     1161364       0.17      0.89      0.28         9
      116570       0.67      1.00      0.80         4
     1176823       0.04      0.83      0.07         6
     1491113       0.33      0.96      0.49        28
     1595929       0.31      0.50      0.38         8
      209233       0.00      0.00      0.00         1
       22930       0.14      1.00      0.25         4
       22956       0.32      0.73      0.44        26
       22961       0.28      0.89      0.42        19
       22967       0.46      0.67      0.55        58
       22973       0.07      0.50      0.13       108
       22983       0.11      0.83      0.20         6
       22985       0.00      0.00      0.00         1
       23150       0.00      0.00      0.00         0
       23154       0.11      0.86      0.19         7
       23158       0.12      0.72      0.21        88
       23176       0.10    

### Save Model

In [1]:
import os

os.makedirs('../../output/models', exist_ok=True)

joblib.dump({'gater': gater, 'specialists': specialists, 'thresholds': best_thresholds}, '../../output/models/hybrid_suite_optimized.pkl')

NameError: name 'joblib' is not defined

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_gater_pred = gater.predict(X_test)

y_tax_test = extract_taxonomy_labels(y_test, species_list, mapping_file=TAXONOMY_PATH)

taxonomy_names = ['Insecta', 'Reptilia', 'Amphibia', 'Mammalia', 'Aves']
print(classification_report(y_tax_test, y_gater_pred, target_names=taxonomy_names))

              precision    recall  f1-score   support

     Insecta       0.67      0.81      0.73       195
    Reptilia       1.00      1.00      1.00         4
    Amphibia       0.76      0.75      0.76       532
    Mammalia       0.80      0.57      0.67        79
        Aves       0.99      0.99      0.99     28122

    accuracy                           0.99     28932
   macro avg       0.85      0.82      0.83     28932
weighted avg       0.99      0.99      0.99     28932

